# Reshape data

In this notebook you'll be needing melt(), pivot(), pivot_table().

Start by importing the data.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("files/world_population.csv")
df.head()

,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area (km²),Density (per km²),Growth Rate,World Population Percentage
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,1.0257,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.9957,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,1.0164,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.9831,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,1.0100,0.00


Transform the data into the following dataframe:

![](files/2023-11-29-14-15-13.png)

In [3]:
# --- Task 1: Melt the population columns ---
# Identify population columns dynamically
pop_cols = [col for col in df.columns if 'Population' in col and col != 'World Population Percentage']

# Melt wide year-based population into long format
world_long = df.melt(id_vars=['Country'], value_vars=pop_cols,
                     var_name='Year_Label', value_name='Population')

# Clean the Year column (remove ' Population' text)
world_long['Year'] = world_long['Year_Label'].str.replace(' Population', '', regex=False)
world_long.drop('Year_Label', axis=1, inplace=True)

print(world_long.head())


          Country  Population  Year
0     Afghanistan    41128771  2022
1         Albania     2842321  2022
2         Algeria    44903225  2022
3  American Samoa       44273  2022
4         Andorra       79824  2022


Starting from your original data, filter out all countries in Europe. Then reshape the data to the following dataframe:

![](files/2023-11-29-14-20-00.png)

The value shown is that of the population in 2022.

In [6]:
# --- Task 2: Pivot back, but filtered for Europe and sorted ---
# Filter for Europe first
europe_data = df[df['Continent'] == 'Europe'].copy()

# Assume years are in columns that are population numbers, not continent/metadata columns.
# Keep Country + population columns only
cols_to_keep = ['Country'] + [col for col in europe_data.columns if col not in ['Country', 'Continent', 'Region', 'Subregion', 'Country Code', 'World Population Percentage']]

europe_data_reduced = europe_data[cols_to_keep]

# Melt to long format first to have year/population rows
europe_long = europe_data_reduced.melt(id_vars=['Country'], var_name='Year_Label', value_name='Population')
# Clean 'Year_Label' to Year
if ' Population' in europe_long['Year_Label'].astype(str).iloc[0]:
    europe_long['Year'] = europe_long['Year_Label'].str.replace(' Population', '', regex=False)
else:
    europe_long['Year'] = europe_long['Year_Label'].astype(str)

# Pivot to wide with countries as rows, years as columns
europe_pivot = europe_long.pivot(index='Country', columns='Year', values='Population')

# Sort by country name
europe_pivot = europe_pivot.sort_index()

print(europe_pivot.head())

Year    1970 Population 1980 Population 1990 Population 2000 Population  \
Country                                                                   
Albania         2324731         2941651         3295066         3182021   
Andorra           19860           35611           53569           66097   
Austria         7465301         7547561         7678729         8010428   
Belarus         9170786         9817257        10428525        10256483   
Belgium         9629376         9828986         9959560        10264343   

Year    2010 Population 2015 Population 2020 Population 2022 Population  \
Country                                                                   
Albania         2913399         2882481         2866849         2842321   
Andorra           71519           71746           77700           79824   
Austria         8362829         8642421         8907777         8939617   
Belarus         9731427         9700609         9633740         9534954   
Belgium        10877947 